# Qwen3 Next 80B A3B in pure MLX

This notebook mirrors the architecture we implement in `mlx_qwen3_next.py` so that you can step through the Qwen3 Next 80B A3B stack with Apple Silicon–friendly MLX tensors. We'll follow the same “致虚极，守静笃” walkthrough used in the README, but every layer now runs in MLX instead of PyTorch.

## Model-at-a-glance

The instruct-tuned 80B A3B release activates three experts per token inside each sparse block. The core transformer alternates three Gated DeltaNet layers with one gated attention layer across 12 groups (48 decoder blocks total), and projects logits for multiple future steps via the multi-token prediction (MTP) head.

In [ ]:
qwen80b_a3b_spec = {
    "total_params": 80_000_000_000,
    "activated_params": 3_000_000_000,
    "layers": 48,
    "hidden_size": 4096,
    "gated_attention": {
        "q_heads": 32,
        "kv_heads": 8,
        "head_dim": 128,
        "rope_dim": 64,
    },
    "gated_deltanet": {
        "v_heads": 32,
        "qk_heads": 8,
        "head_dim": 128,
    },
    "moe": {
        "experts": 512,
        "active_experts": 3,
        "shared_experts": 1,
        "intermediate_dim": 14336,
    },
    "context": {
        "native": 262_144,
        "max_tested": 1_010_000,
    },
    "mtp_predictions": 4,
}

qwen80b_a3b_spec

## Imports and MLX runtime

Everything below depends on the MLX runtime. If you run this notebook on a non-macOS host, install the pre-built `mlx` wheel (or comment out the `require_mlx()` line and stub the layers). We also pull helpers straight from `mlx_qwen3_next.py` so that the notebook uses the exact same implementation as the CLI loader.

In [ ]:
from __future__ import annotations

from typing import Optional

try:
    from transformers import AutoTokenizer  # optional, for full tokenizer parity
except Exception:
    AutoTokenizer = None

from mlx_qwen3_next import (
    DEFAULT_MODEL_ID,
    Qwen3NextConfig,
    Qwen3NextForCausalLM,
    Qwen3NextDecoderLayer,
    Qwen3NextGatedDeltaNet,
    Qwen3NextAttention,
    build_causal_mask,
    require_mlx,
)

require_mlx()

import mlx.core as mx
import mlx.nn as nn

## Tokenising “致虚极，守静笃”

We'll keep the playful four-character phrase that shows up throughout the tutorial. To stay focused on the core math, we build a miniature character-level vocabulary and emit MLX arrays—feel free to swap in the official tokenizer by uncommenting the optional `transformers` cell.

In [ ]:
phrase = "致虚极，守静笃"

symbols = list(dict.fromkeys(phrase))
vocab = {"<pad>": 0, "<unk>": 1}
for idx, symbol in enumerate(symbols, start=2):
    vocab[symbol] = idx

token_ids = [vocab.get(ch, vocab["<unk>"]) for ch in phrase]
mx_tokens = mx.array(token_ids, dtype=mx.int32)[None, :]

print("Characters → token ids:")
for ch, idx in zip(phrase, token_ids):
    print(f"  {ch} → {idx}")
print("Tensor shape:", mx_tokens.shape)

### (Optional) Using the official tokenizer

If you want byte-level compatibility with the released checkpoint, run the cell below. It will download the tokenizer from Hugging Face the first time you execute it.

In [ ]:
if AutoTokenizer is not None:
    tokenizer = AutoTokenizer.from_pretrained(
        DEFAULT_MODEL_ID, trust_remote_code=True, padding_side="left"
    )
    hf_tokens = tokenizer(phrase, return_tensors="np")
    mx_tokens = mx.array(hf_tokens["input_ids"], dtype=mx.int32)
    print("Tokenizer-loaded ids:", mx_tokens)
else:
    print("Install `transformers` to mirror the official tokenizer.")

## Building a toy Qwen3 Next stack in MLX

The full 80B checkpoint is massive, so we'll shrink the configuration while keeping the same ratios: each group contains three linear Gated DeltaNet blocks followed by one attention block. That pattern is controlled by `full_attention_interval=4` in the config.

In [ ]:
toy_config = Qwen3NextConfig(
    vocab_size=len(vocab),
    hidden_size=256,
    num_hidden_layers=12,
    num_attention_heads=8,
    num_key_value_heads=4,
    linear_num_value_heads=16,
    linear_num_key_heads=4,
    head_dim=32,
    linear_key_head_dim=32,
    linear_value_head_dim=16,
    intermediate_size=1024,
    moe_intermediate_size=1024,
    shared_expert_intermediate_size=1024,
    num_experts=0,
    num_experts_per_tok=0,
    full_attention_interval=4,
    tie_word_embeddings=True,
    rope_theta=1_000_000.0,
    partial_rotary_factor=0.5,
)

model = Qwen3NextForCausalLM(toy_config)
print(f"Embedding matrix shape: {model.model.embed_tokens.weight.shape}")
print(f"Decoder layers: {len(model.layers)}")

### Verifying the 3×DeltaNet + 1×Attention cadence

Each decoder layer toggles between a linear Gated DeltaNet update and a gated self-attention block. With `full_attention_interval=4`, layers 1–3 are DeltaNet and layer 4 is attention, repeating all the way down.

In [ ]:
group_plan = []
for idx, layer in enumerate(model.layers):
    label = 'Gated DeltaNet' if layer.is_linear else 'Gated Attention'
    group_plan.append(f'Layer {idx + 1:02d}: {label}')
print('
'.join(group_plan))

## Running tokens through the stack

With the toy configuration in place we can follow the embeddings as they flow through the twelve decoder layers. MLX keeps tensors lazy, so we call `mx.eval` on the residual deltas to materialise the numbers for inspection.

In [ ]:
mx.random.seed(0)
inputs = mx.asarray(mx_tokens)
mask = build_causal_mask(inputs.shape[1], mx.float32)

def describe_step(name: str, delta: mx.array) -> None:
    magnitude = mx.mean(mx.abs(delta)).item()
    print(f'  {name:<18} Δ|x| ≈ {magnitude:.6f}')

hidden = model.model.embed_tokens(inputs)
print('Embedding output:', hidden.shape)
for idx, layer in enumerate(model.layers):
    pre = hidden
    hidden = layer(hidden, mask=mask)
    describe_step(group_plan[idx], hidden - pre)

encoded = model.model.norm(hidden)
print('
Post-norm representation:', encoded.shape)

## Multi-token prediction head in MLX

Qwen3 Next predicts several future tokens at once. The production checkpoint mixes four projections and shifts them so that prediction 0 targets the next token, prediction 1 targets the token two steps out, and so on. Here's a compact MLX version of that head plus a helper loss you can reuse for training experiments.

In [ ]:
class MultiTokenPredictionHead(nn.Module):
    def __init__(self, hidden_size: int, vocab_size: int, predictions: int = 4):
        super().__init__()
        self.predictions = predictions
        self.vocab_size = vocab_size
        self.proj = nn.Linear(hidden_size, vocab_size * predictions, bias=False)

    def __call__(self, hidden: mx.array) -> mx.array:
        batch, seq, _ = hidden.shape
        logits = self.proj(hidden).reshape(batch, seq, self.predictions, self.vocab_size)
        outputs = []
        for offset in range(self.predictions):
            step = logits[:, :, offset]
            if offset:
                step = step[:, :-offset]
                step = mx.pad(step, ((0, 0), (offset, 0), (0, 0)))
            outputs.append(step)
        return mx.stack(outputs, axis=1)


def shift_targets(targets: mx.array, offset: int) -> mx.array:
    if offset == 0:
        return targets
    pad = mx.full((targets.shape[0], offset), -100, dtype=targets.dtype)
    return mx.concatenate([pad, targets[:, :-offset]], axis=1)


def masked_cross_entropy(logits: mx.array, targets: mx.array) -> mx.array:
    mask = targets != -100
    log_probs = logits - mx.logsumexp(logits, axis=-1, keepdims=True)
    gathered = mx.take_along_axis(log_probs, targets[..., None], axis=-1).squeeze(-1)
    loss = -(gathered * mask)
    return loss.sum() / mask.sum().astype(mx.float32)


def multi_token_cross_entropy(
    logits: mx.array, targets: mx.array, *, multi_token_weight: float = 0.4
) -> mx.array:
    predictions = logits.shape[1]
    losses = []
    for offset in range(predictions):
        shifted = shift_targets(targets, offset)
        losses.append(masked_cross_entropy(logits[:, offset], shifted))
    main = losses[0]
    if predictions == 1:
        return main
    aux = mx.stack(losses[1:]).mean()
    return main * (1.0 - multi_token_weight) + aux * multi_token_weight


mtp_head = MultiTokenPredictionHead(toy_config.hidden_size, toy_config.vocab_size)
logits = mtp_head(encoded)
mx.eval(logits)
print('Logits tensor shape:', logits.shape)
print('Prediction tiers:', logits[:, :, 0, :3])

loss = multi_token_cross_entropy(logits, mx.asarray(mx_tokens))
print('Sample multi-token loss:', float(loss))